In [ ]:
from PDESolver import *

## Example 1: Heat Equation - Laplacian Operator

In [ ]:
# Symbolic variables
x = symbols('x', real=True)
xi = symbols('xi', real=True)
t_sym = symbols('t', real=True, positive=True)

# Laplacian symbol: Δ = -∂²/∂x²  →  symbol = -ξ²
laplacian_symbol = -xi**2
Laplacian = PseudoDifferentialOperator(laplacian_symbol, [x], mode='symbol')

print("\nLaplacian symbol p(x,ξ) =")
pprint(Laplacian.symbol)

# Compute exp(t*Δ) for the heat equation: u_t = Δu
print("\nComputing exp(t*Δ) up to order 3...")
heat_kernel_symbol = Laplacian.exponential_symbol(t=t_sym, order=3)
print("\nHeat kernel symbol (order 3):")
pprint(simplify(heat_kernel_symbol))

# Exact solution for comparison
exact_heat_kernel = exp(-t_sym * xi**2)
print("\nExact heat kernel symbol:")
pprint(exact_heat_kernel)

# Difference
print("\nDifference (should be small for small t):")
diff = simplify(heat_kernel_symbol - exact_heat_kernel)
pprint(diff)

# Numerical comparison loop
t_values = [0.05, 0.1, 0.15, 0.2, 0.3]
xi_vals = np.linspace(-5, 5, 200)

print("\n" + "-"*70)
print("Numerical comparisons for various time steps:")
print("-"*70)

errors = []  # to store errors

for t_val in t_values:
    print(f"\nComparing at t = {t_val} ...")

    # Numerical lambdify functions
    heat_approx_func = lambdify(xi, heat_kernel_symbol.subs(t_sym, t_val), 'numpy')
    heat_exact_func = lambdify(xi, exact_heat_kernel.subs(t_sym, t_val), 'numpy')

    # Evaluate
    heat_approx = heat_approx_func(xi_vals)
    heat_exact = heat_exact_func(xi_vals)

    # Compute error
    error = np.max(np.abs(heat_approx.real - heat_exact.real))
    errors.append(error)

    # Plot
    plt.figure(figsize=(10, 6))
    plt.plot(xi_vals, heat_exact.real, 'b-', label='Exact: exp(-tξ²)', linewidth=2)
    plt.plot(xi_vals, heat_approx.real, 'r--', label='Asymptotic (order 3)', linewidth=2)
    plt.xlabel('ξ (frequency)', fontsize=12)
    plt.ylabel('Symbol value', fontsize=12)
    plt.title(f'Heat Kernel Symbol at t={t_val}', fontsize=14)
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f"Maximum error at t={t_val}: {error:.2e}")

# Global summary
print("\n" + "-"*70)
print("Summary of maximum errors:")
print("-"*70)
for t_val, err in zip(t_values, errors):
    print(f"t = {t_val:.2f}  →  max error = {err:.3e}")

# Optional: plot error vs time
plt.figure(figsize=(8, 5))
plt.plot(t_values, errors, 'o-', linewidth=2)
plt.xlabel('t', fontsize=12)
plt.ylabel('Max error |approx - exact|', fontsize=12)
plt.title('Error evolution vs time (asymptotic order 3)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Example 2: Schrödinger Equation - Harmonic Oscillator

In [ ]:
# ================================================================
# Example: Quantum harmonic oscillator propagator
# ================================================================

print("\n" + "="*70)
print("Example: Schrödinger Propagator for Harmonic Oscillator")
print("="*70)

# Symbolic variables
x = symbols('x', real=True)
xi = symbols('xi', real=True)
t_sym = symbols('t', real=True, positive=True)

# Hamiltonian symbol H(x, ξ) = (ξ² + x²)/2
H_symbol = (xi**2 + x**2) / 2
Hamiltonian = PseudoDifferentialOperator(H_symbol, [x], mode='symbol')

print("\nHamiltonian symbol H(x,ξ) =")
pprint(Hamiltonian.symbol)

# Compute the asymptotic expansion of exp(-i t H)
print("\nComputing U(t) = exp(-i t H) up to order 4...")
propagator_symbol = Hamiltonian.exponential_symbol(t=-I*t_sym, order=4)

print("\nAsymptotic propagator symbol (order 4):")
pprint(simplify(propagator_symbol))

# Exact propagator symbol for comparison: exp(-i t (ξ² + x²)/2)
exact_propagator = exp(-I * t_sym * (xi**2 + x**2) / 2)

print("\nExact propagator symbol:")
pprint(exact_propagator)

# Compare asymptotic vs exact
print("\nDifference (should be small for small t):")
diff_symbol = simplify(propagator_symbol - exact_propagator)
pprint(diff_symbol)

# ================================================================
# Numerical evaluation and visualization
# ================================================================
import numpy as np
import matplotlib.pyplot as plt

t_values = [0.05, 0.1, 0.2]   # small to moderate times
x_val = 0.0                   # fix position
xi_vals = np.linspace(-4, 4, 400)

# Prepare lambdified functions
prop_asym_func = lambdify((xi, t_sym), propagator_symbol.subs(x, x_val), 'numpy')
prop_exact_func = lambdify((xi, t_sym), exact_propagator.subs(x, x_val), 'numpy')

for t_val in t_values:
    print("\n" + "-"*70)
    print(f"Numerical comparison at t = {t_val}")
    print("-"*70)

    prop_asym = prop_asym_func(xi_vals, t_val)
    prop_exact = prop_exact_func(xi_vals, t_val)

    # Compute errors on real and imaginary parts
    err_real = np.max(np.abs(prop_asym.real - prop_exact.real))
    err_imag = np.max(np.abs(prop_asym.imag - prop_exact.imag))
    print(f"Max real error  : {err_real:.2e}")
    print(f"Max imag error  : {err_imag:.2e}")

    # Plot results
    plt.figure(figsize=(10, 6))
    plt.plot(xi_vals, prop_exact.real, 'b-', label='Re[exact]', linewidth=2)
    plt.plot(xi_vals, prop_asym.real, 'r--', label='Re[asymptotic]', linewidth=2)
    plt.plot(xi_vals, prop_exact.imag, 'g-', label='Im[exact]', linewidth=2)
    plt.plot(xi_vals, prop_asym.imag, 'm--', label='Im[asymptotic]', linewidth=2)
    plt.xlabel('ξ (momentum)', fontsize=12)
    plt.ylabel('Propagator value', fontsize=12)
    plt.title(f'Harmonic oscillator propagator at t = {t_val}, x = {x_val}', fontsize=14)
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# ================================================================
# Unitarity check
# ================================================================
print("\n" + "="*70)
print("Unitarity check: |U(t)|² ≈ 1")
print("="*70)

prop_unitarity = simplify(propagator_symbol * conjugate(propagator_symbol))
print("\n|U(t)|² (symbolic):")
pprint(prop_unitarity)

# Evaluate numerically for small t
t_test = 0.1
prop_unit_func = lambdify((xi, t_sym), prop_unitarity.subs(x, x_val), 'numpy')
vals_unit = prop_unit_func(xi_vals, t_test)

plt.figure(figsize=(9, 5))
plt.plot(xi_vals, vals_unit.real, 'k-', linewidth=2, label='|U(t)|²')
plt.xlabel('ξ (momentum)', fontsize=12)
plt.ylabel('|U(t)|²', fontsize=12)
plt.title(f'Unitarity check at t={t_test}', fontsize=14)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

err_unit = np.max(np.abs(vals_unit.real - 1))
print(f"Maximum deviation from unitarity: {err_unit:.2e}")


## Example 3: Transport/Advection Operator

In [ ]:
# --- Symbolic setup ---
x, xi, t_sym = symbols('x xi t', real=True)
c = symbols('c', real=True, positive=True)

# Symbol: p(x, ξ) = i * c * ξ  (transport avec vitesse c)
transport_symbol = I * c * xi
Transport = PseudoDifferentialOperator(transport_symbol, [x], mode='symbol')

print("\nTransport operator symbol p(x,ξ) =")
pprint(Transport.symbol)

# --- Exponentielle symbolique ---
print("\nComputing exp(t * i*c*∂_x) up to order 5...")
shift_symbol = Transport.exponential_symbol(t=t_sym, order=5)
shift_symbol_simplified = simplify(shift_symbol)

print("\nAsymptotic shift operator symbol (order 5):")
pprint(shift_symbol_simplified)

# --- Solution exacte ---
exact_shift = exp(I * c * t_sym * xi)
print("\nExact shift operator symbol:")
pprint(exact_shift)

# --- Différence symbolique ---
diff_symbol = simplify(shift_symbol_simplified - exact_shift)
print("\nDifference (should be exactly 0):")
pprint(diff_symbol)

# ============================================================
# Numerical comparison for various times
# ============================================================

t_values = [0.1, 0.5, 1.0, 2.0, 3.0]
xi_vals = np.linspace(-5, 5, 400)
c_val = 1.0

# Lambdify symbolic expressions
shift_approx_func = lambdify((xi, t_sym, c), shift_symbol_simplified, 'numpy')
shift_exact_func  = lambdify((xi, t_sym, c), exact_shift, 'numpy')

errors = []

print("\n" + "-"*70)
print("Numerical comparisons for various time steps:")
print("-"*70)

for t_val in t_values:
    print(f"\nComparing at t = {t_val} ...")

    # Evaluate numerical values
    approx_vals = shift_approx_func(xi_vals, t_val, c_val)
    exact_vals = shift_exact_func(xi_vals, t_val, c_val)

    # Compute difference
    error = np.max(np.abs(approx_vals - exact_vals))
    errors.append(error)

    # Plot real and imaginary parts
    fig, axes = plt.subplots(2, 1, figsize=(10, 8))

    axes[0].plot(xi_vals, np.real(exact_vals), 'b-', label='Exact', linewidth=2)
    axes[0].plot(xi_vals, np.real(approx_vals), 'r--', label='Asymptotic (order 5)', linewidth=2)
    axes[0].set_ylabel('Re[Shift symbol]', fontsize=12)
    axes[0].set_title(f'Real part of exp(i c t ξ) at t={t_val}', fontsize=14)
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    axes[1].plot(xi_vals, np.imag(exact_vals), 'b-', label='Exact', linewidth=2)
    axes[1].plot(xi_vals, np.imag(approx_vals), 'r--', label='Asymptotic (order 5)', linewidth=2)
    axes[1].set_xlabel('ξ (frequency)', fontsize=12)
    axes[1].set_ylabel('Im[Shift symbol]', fontsize=12)
    axes[1].set_title('Imaginary part', fontsize=14)
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    print(f"Maximum error at t={t_val}: {error:.2e}")

# ============================================================
# Summary of errors
# ============================================================

print("\n" + "-"*70)
print("Summary of maximum errors:")
print("-"*70)
for t_val, err in zip(t_values, errors):
    print(f"t = {t_val:.2f}  →  max error = {err:.3e}")

plt.figure(figsize=(8, 5))
plt.plot(t_values, errors, 'o-', linewidth=2)
plt.xlabel('t', fontsize=12)
plt.ylabel('Max error |approx - exact|', fontsize=12)
plt.title('Error evolution vs time (Transport operator)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Example 4: Convergence with Order

In [ ]:
from sympy import symbols, exp, lambdify, simplify, pprint
import numpy as np
import matplotlib.pyplot as plt

# --- Symbolic setup ---
x, xi, t_sym = symbols('x xi t', real=True)
mixed_symbol = xi**2 + x*xi + x**2
Mixed = PseudoDifferentialOperator(mixed_symbol, [x], mode='symbol')

print("\nMixed operator symbol p(x,ξ) =")
pprint(Mixed.symbol)

# --- Parameters ---
x_val = 1.0
xi_vals = np.linspace(-2, 2, 200)
t_values = [0.05, 0.1, 0.2, 0.5]
orders = [1, 2, 3, 4, 5]

# --- Compute all exponentials ---
exponentials = {}
for t_val in t_values:
    exponentials[t_val] = {}
    for ord in orders:
        exp_sym = Mixed.exponential_symbol(t=t_val, order=ord)
        exponentials[t_val][ord] = simplify(exp_sym)

# --- Visualization loop ---
for t_val in t_values:
    plt.figure(figsize=(12, 8))
    for ord in orders:
        exp_func = lambdify((xi,), exponentials[t_val][ord].subs(x, x_val), 'numpy')
        exp_vals = exp_func(xi_vals)
        plt.plot(xi_vals, exp_vals.real, linewidth=2, label=f'Order {ord}')
    plt.xlabel('ξ', fontsize=12)
    plt.ylabel('Re[exp(tP)]', fontsize=12)
    plt.title(f'Convergence of exp(tP) at x={x_val}, t={t_val}', fontsize=14)
    plt.legend(fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# --- Convergence analysis ---
print("\n" + "-"*70)
print("Convergence analysis across orders and times:")
print("-"*70)
for t_val in t_values:
    print(f"\nAt t = {t_val}:")
    for i in range(len(orders)-1):
        ord1, ord2 = orders[i], orders[i+1]
        func1 = lambdify(xi, exponentials[t_val][ord1].subs(x, x_val), 'numpy')
        func2 = lambdify(xi, exponentials[t_val][ord2].subs(x, x_val), 'numpy')
        vals1, vals2 = func1(xi_vals), func2(xi_vals)
        diff = np.max(np.abs(vals1 - vals2))
        print(f"  Max difference between order {ord1} and {ord2}: {diff:.2e}")


## Example 5: 2D Heat Equation

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
# --- 2D Symbols
x, y = symbols('x y', real=True)
xi, eta = symbols('xi eta', real=True)
t = symbols('t', real=True, positive=True)
# --- 2D Laplacian
laplacian_2d_symbol = -(xi**2 + eta**2)
Laplacian2D = PseudoDifferentialOperator(laplacian_2d_symbol, [x, y], mode='symbol')
print("\n2D Laplacian symbol p(x,y,ξ,η) =")
print(laplacian_2d_symbol)
# ============================================================
# Loops: Asymptotic orders and times
# ============================================================
orders = [1, 2, 3, 4]
t_values = [0.01, 0.05, 0.1, 0.2]
x_val, y_val = 0.0, 0.0
xi_vals = np.linspace(-4, 4, 80)
eta_vals = np.linspace(-4, 4, 80)
XI, ETA = np.meshgrid(xi_vals, eta_vals)
# Exact function
exact_symbol = exp(-t * (xi**2 + eta**2))
exact_func = lambdify((xi, eta, t), exact_symbol, 'numpy')
errors_Linf = np.zeros((len(orders), len(t_values)))
errors_L2 = np.zeros_like(errors_Linf)
# ============================================================
# Main loop
# ============================================================
for i_ord, order in enumerate(orders):
    print(f"\n=== Order {order} ===")
    for j_t, t_val in enumerate(t_values):
        approx_symbol = Laplacian2D.exponential_symbol(t=t_val, order=order)
        approx_func = lambdify((xi, eta),
                               approx_symbol.subs([(x, x_val), (y, y_val)]),
                               'numpy')
        approx_vals = approx_func(XI, ETA).real
        exact_vals = exact_func(XI, ETA, t_val).real
        err = np.abs(approx_vals - exact_vals)
        errors_Linf[i_ord, j_t] = np.max(err)
        errors_L2[i_ord, j_t] = np.sqrt(np.mean(err**2))
        # --- Visualization for a given t
        if j_t == 1:  # Only one figure per order for an intermediate t
            fig, axes = plt.subplots(1, 3, figsize=(15, 4))
            fig.suptitle(f"exp(tΔ) in 2D — order {order}, t={t_val}", fontsize=14)
            im0 = axes[0].imshow(approx_vals, extent=[-4, 4, -4, 4], origin='lower', cmap='viridis')
            axes[0].set_title("Asymptotic (Re part)")
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            im1 = axes[1].imshow(exact_vals, extent=[-4, 4, -4, 4], origin='lower', cmap='viridis')
            axes[1].set_title("Exact (Re part)")
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            im2 = axes[2].imshow(err, extent=[-4, 4, -4, 4], origin='lower', cmap='Reds')
            axes[2].set_title("|Error|")
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            for ax in axes:
                ax.set_xlabel("ξ")
                ax.set_ylabel("η")
            plt.tight_layout()
            plt.show()
# ============================================================
# Global error analysis
# ============================================================
plt.figure(figsize=(10, 6))
for i, order in enumerate(orders):
    plt.plot(t_values, errors_Linf[i, :], 'o-', label=f'Order {order}')
plt.xlabel('t')
plt.ylabel('L∞ error')
plt.title('Convergence of exp(tΔ) in 2D — Max error vs time')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
plt.figure(figsize=(10, 6))
for i, order in enumerate(orders):
    plt.plot(t_values, errors_L2[i, :], 's-', label=f'Order {order}')
plt.xlabel('t')
plt.ylabel('L² error')
plt.title('Convergence of exp(tΔ) in 2D — L² error vs time')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
# ============================================================
# Synthetic error summary
# ============================================================
print("\nSummary of errors:")
print("Order |   t   |  L∞ Error   |   L² Error")
print("------------------------------------------")
for i, order in enumerate(orders):
    for j, t_val in enumerate(t_values):
        print(f"{order:>5d} | {t_val:4.2f} | {errors_Linf[i,j]:10.2e} | {errors_L2[i,j]:10.2e}")


## Example 6: Heisenberg Algebra and Canonical Commutation Relations

In [ ]:
x = symbols('x', real=True)
xi = symbols('xi', real=True)
# Generators of the Heisenberg algebra
# Position: Q with symbol x
# Momentum: P with symbol ξ
# Identity: I with symbol 1
Q_symbol = x
P_symbol = xi
Q_op = PseudoDifferentialOperator(Q_symbol, [x], mode='symbol')
P_op = PseudoDifferentialOperator(P_symbol, [x], mode='symbol')
print("\nPosition operator Q, symbol:")
pprint(Q_op.symbol)
print("\nMomentum operator P, symbol:")
pprint(P_op.symbol)
# Compute the commutator [Q, P] via asymptotic composition
print("\n" + "-"*70)
print("Computing [Q, P] = QP - PQ")
print("-"*70)
# QP via composition
QP_symbol = Q_op.compose_asymptotic(P_op, order=3)
print("\nQP symbol (order 3):")
pprint(simplify(QP_symbol))
# PQ via composition
PQ_symbol = P_op.compose_asymptotic(Q_op, order=3)
print("\nPQ symbol (order 3):")
pprint(simplify(PQ_symbol))
# Commutator [Q,P]
commutator_QP = simplify(QP_symbol - PQ_symbol)
print("\n[Q, P] symbol:")
pprint(commutator_QP)
print("\n✓ Expected: [Q, P] = iℏ (with ℏ=1 in our units)")
print(f"✓ Obtained: {commutator_QP}")
# Verify the canonical commutation relation
if commutator_QP == 1.0*I:
    print("\n✅ CANONICAL COMMUTATION RELATION VERIFIED!")
else:
    print(f"\n⚠️  Expected I, got {commutator_QP}")
# Compute the commutator [P, Q] via asymptotic composition
print("\n" + "-"*70)
print("Computing [P, Q] = PQ - QP")
print("-"*70)
# QP via composition
QP_symbol = Q_op.compose_asymptotic(P_op, order=3)
print("\nQP symbol (order 3):")
pprint(simplify(QP_symbol))
# PQ via composition
PQ_symbol = P_op.compose_asymptotic(Q_op, order=3)
print("\nPQ symbol (order 3):")
pprint(simplify(PQ_symbol))
# Commutator [P,Q]
commutator_PQ = simplify(PQ_symbol - QP_symbol)
print("\n[P, Q] symbol:")
pprint(commutator_PQ)
print("\n✓ Expected: [P, Q] = -iℏ (with ℏ=1 in our units)")
print(f"✓ Obtained: {commutator_PQ}")
# Verify the canonical commutation relation
if commutator_PQ == -1.0*I:
    print("\n✅ CANONICAL COMMUTATION RELATION VERIFIED!")
else:
    print(f"\n⚠️  Expected I, got {commutator_QP}")
print("\n" + "="*70)
print("Heisenberg Algebra Structure Constants")
print("="*70)
# The Heisenberg algebra has the structure:
# [P, Q] = -iI
# [P, I] = 0
# [Q, I] = 0
print("""
Heisenberg algebra generators: {P, Q, I}
Structure relations:
  [Q, P] = iI
  [P, Q] = -iI
  [Q, I] = 0
  [P, I] = 0
  [I, I] = 0
This is a 3-dimensional nilpotent Lie algebra.
""")


## Example 7: Baker-Campbell-Hausdorff Formula for Heisenberg Group

In [ ]:
# For two operators X, Y, the BCH formula gives:
# log(exp(X) exp(Y)) = X + Y + (1/2)[X,Y] + (1/12)[X,[X,Y]] - (1/12)[Y,[X,Y]] + ...
# Let X = aQ and Y = bP
a, b = symbols('a b', real=True)
t = symbols('t', real=True)
X_symbol = a * x
Y_symbol = b * xi
X_op = PseudoDifferentialOperator(X_symbol, [x], mode='symbol')
Y_op = PseudoDifferentialOperator(Y_symbol, [x], mode='symbol')
print(f"\nX = aQ, symbol: {X_op.symbol}")
print(f"Y = bP, symbol: {Y_op.symbol}")

# Commutator [X,Y] = [aQ, bP] = ab[Q,P] = iab
commutator_XY = X_op.commutator_symbolic(Y_op, order=3)
print(f"\n[X, Y] = [aQ, bP] =")
pprint(commutator_XY)

# Compute exp(tX) and exp(tY)
print("\n" + "-"*70)
print("Computing exp(tX) and exp(tY)")
print("-"*70)
exp_tX = X_op.exponential_symbol(t=t, order=4)
exp_tY = Y_op.exponential_symbol(t=t, order=4)
print("\nexp(tX) symbol:")
pprint(simplify(exp_tX))
print("\nexp(tY) symbol:")
pprint(simplify(exp_tY))

# Composition exp(tX) ∘ exp(tY)
print("\n" + "-"*70)
print("Computing exp(tX) ∘ exp(tY) via composition")
print("-"*70)
exp_tX_op = PseudoDifferentialOperator(exp_tX, [x], mode='symbol')
exp_tY_op = PseudoDifferentialOperator(exp_tY, [x], mode='symbol')
product_symbol = exp_tY_op.compose_asymptotic(exp_tX_op, order=3, mode='weyl')
print("\nexp(tX) ∘ exp(tY) symbol:")
pprint(simplify(product_symbol))

# Exact BCH formula for the Heisenberg group
# exp(aQ) exp(bP) = exp(aQ + bP + (ab/2)[Q,P])
#                 = exp(aQ + bP + iab/2)
print("\n" + "-"*70)
print("Baker-Campbell-Hausdorff formula for Heisenberg group")
print("-"*70)
print("""
For the Heisenberg group, the BCH formula terminates at second order:
exp(aQ) exp(bP) = exp(aQ + bP + (ab/2)i)
This is because [Q,P] = iI commutes with everything, so higher
commutators vanish.
Physical interpretation:
- This is the quantum mechanical composition of translation operators
- The phase factor exp(iab/2) is the Weyl ordering correction
""")

# Numerical verification
print("\n" + "-"*70)
print("Numerical verification")
print("-"*70)
a_val, b_val, t_val = 1.0, 0.5, 1.0
x_val = 0.0
product_func = lambdify(xi, product_symbol.subs([(a, a_val), (b, b_val),
                                                   (t, t_val), (x, x_val)]), 'numpy')
# Exact BCH solution
BCH_exact = exp(t_val * (a_val * x + b_val * xi + I * a_val * b_val / 2))
BCH_func = lambdify(xi, BCH_exact.subs(x, x_val), 'numpy')
xi_vals = np.linspace(-3, 3, 100)
product_vals = product_func(xi_vals)
BCH_vals = BCH_func(xi_vals)
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(xi_vals, product_vals.real, 'b-', label='Composition', linewidth=2)
plt.plot(xi_vals, BCH_vals.real, 'r--', label='BCH exact', linewidth=2)
plt.xlabel('ξ')
plt.ylabel('Real part')
plt.title('exp(aQ) ∘ exp(bP) - Real part')
plt.legend()
plt.grid(True, alpha=0.3)
plt.subplot(1, 2, 2)
plt.plot(xi_vals, product_vals.imag, 'b-', label='Composition', linewidth=2)
plt.plot(xi_vals, BCH_vals.imag, 'r--', label='BCH exact', linewidth=2)
plt.xlabel('ξ')
plt.ylabel('Imaginary part')
plt.title('exp(aQ) ∘ exp(bP) - Imaginary part')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
error = np.max(np.abs(product_vals - BCH_vals))
print(f"\nMaximum error: {error:.2e}")


## Example 8: Geodesic Flow on Riemannian Manifolds

In [ ]:
x, y = symbols('x y', real=True)
xi, eta = symbols('xi eta', real=True)
print("""
CONTEXT: Differential Geometry
On a Riemannian manifold (M, g), the geodesic flow is generated by
the Hamiltonian H(x, ξ) = (1/2) g^{ij}(x) ξ_i ξ_j
For a surface of revolution with metric:
  ds² = dx² + f(x)² dy²
The Hamiltonian is:
  H(x, y, ξ, η) = (1/2)(ξ² + η²/f(x)²)
""")
# Example: surface of revolution with f(x) = 1 + x²
f = 1 + x**2
H_geodesic = (xi**2 + eta**2 / f**2) / 2
Geodesic_H = PseudoDifferentialOperator(H_geodesic, [x, y], mode='symbol')
print("\nGeodesic Hamiltonian H(x, y, ξ, η) =")
pprint(simplify(Geodesic_H.symbol))
# Hamiltonian flow (Hamilton's equations)
print("\n" + "-"*70)
print("Hamiltonian flow equations:")
print("-"*70)
flow = Geodesic_H.symplectic_flow()
print("\nSymplectic flow:")
for key, val in flow.items():
    print(f"{key} = ", end="")
    pprint(simplify(val))
print("""
These equations describe how geodesics evolve on the surface.
Physical interpretation (if this were a mechanical system):
- x, y: position on the surface
- ξ, η: momentum/velocity covectors
- The flow preserves the Hamiltonian (energy conservation)
""")
# Time evolution of the geodesic flow
t = symbols('t', real=True, positive=True)
print("\n" + "-"*70)
print("Time evolution operator exp(tH) for geodesic flow")
print("-"*70)
geodesic_propagator = Geodesic_H.exponential_symbol(t=t, order=2)
print("\nGeodesic propagator (order 3):")
pprint(simplify(geodesic_propagator))
print("""
This operator propagates initial conditions along geodesics.
Applications:
- Ray tracing in geometric optics
- Minimal surfaces
- General relativity (null geodesics = light rays)
""")
# Visualization of the geodesic flow
print("\n" + "-"*70)
print("Visualization: Phase space portrait")
print("-"*70)
from scipy.integrate import odeint
def geodesic_flow_ode(state, t, f_func):
    """ODE system for geodesic flow"""
    x_val, y_val, xi_val, eta_val = state

    f_val = f_func(x_val)
    df_dx = 2*x_val  # derivative of f = 1 + x²

    dx_dt = xi_val
    dy_dt = eta_val / f_val**2
    dxi_dt = eta_val**2 * df_dx / f_val**3
    deta_dt = 0  # η is conserved (symmetry in y)

    return [dx_dt, dy_dt, dxi_dt, deta_dt]
# Initial conditions
x0, y0 = 0.0, 0.0
xi0, eta0 = 1.0, 0.5
initial_state = [x0, y0, xi0, eta0]
t_span = np.linspace(0, 2, 100)
f_func = lambda x: 1 + x**2
trajectory = odeint(geodesic_flow_ode, initial_state, t_span, args=(f_func,))
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
# Position space (x, y)
axes[0, 0].plot(trajectory[:, 0], trajectory[:, 1], 'b-', linewidth=2)
axes[0, 0].plot(x0, y0, 'go', markersize=10, label='Start')
axes[0, 0].plot(trajectory[-1, 0], trajectory[-1, 1], 'ro', markersize=10, label='End')
axes[0, 0].set_xlabel('x')
axes[0, 0].set_ylabel('y')
axes[0, 0].set_title('Geodesic on Surface')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)
# Phase space (x, ξ)
axes[0, 1].plot(trajectory[:, 0], trajectory[:, 2], 'r-', linewidth=2)
axes[0, 1].set_xlabel('x')
axes[0, 1].set_ylabel('ξ')
axes[0, 1].set_title('Phase Space (x, ξ)')
axes[0, 1].grid(True, alpha=0.3)
# Phase space (y, η)
axes[1, 0].plot(trajectory[:, 1], trajectory[:, 3], 'g-', linewidth=2)
axes[1, 0].set_xlabel('y')
axes[1, 0].set_ylabel('η')
axes[1, 0].set_title('Phase Space (y, η) - η conserved!')
axes[1, 0].grid(True, alpha=0.3)
# Hamiltonian conservation
H_values = 0.5 * (trajectory[:, 2]**2 + trajectory[:, 3]**2 / (1 + trajectory[:, 0]**2)**2)
axes[1, 1].plot(t_span, H_values, 'k-', linewidth=2)
axes[1, 1].set_xlabel('t')
axes[1, 1].set_ylabel('H (Energy)')
axes[1, 1].set_title('Hamiltonian Conservation')
axes[1, 1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f"\nEnergy drift: {np.std(H_values):.2e}")
print("✓ Hamiltonian is conserved (symplectic flow)")


## Example 9: Statistical Mechanics - Fokker-Planck Equation

In [ ]:
x = symbols('x', real=True)
xi = symbols('xi', real=True)
# Temperature and physical parameters
kB, T, gamma, m = symbols('k_B T gamma m', real=True, positive=True)
print("""
CONTEXT: Statistical Mechanics
The Fokker-Planck equation describes the evolution of a probability
density in phase space for a particle in a heat bath:
∂ρ/∂t = L_FP ρ
where L_FP is the Fokker-Planck operator (generator).
For a Brownian particle with friction γ and temperature T:
L_FP = -(ξ/m)∂/∂x + γ∂/∂ξ(ξ + kT∂/∂ξ)
In symbol form, this becomes a pseudo-differential operator.
""")
# Kramers operator (Fokker-Planck)
# Simplified version with m = kB = 1, T and γ as parameters
# Drift term: -(ξ/m)∂/∂x  →  symbol: -(ξ/m)·iξ = -iξ²/m
# Friction term: γξ∂/∂ξ  → symbol: γξ·iξ = iγξ²
# Diffusion term: γkT∂²/∂ξ²  → symbol: -γkT·ξ²
# Simplified version (dimensionless)
m_val = 1
kB_val = 1
# L_FP symbol (main part)
# Drift: -ξ·ix (advection in x)
# Friction: γξ·iξ
# Diffusion: -γT·ξ²
L_FP_symbol = -I*xi**2 + I*gamma*xi**2 - gamma*T*xi**2
print("\nFokker-Planck operator symbol (simplified):")
pprint(L_FP_symbol)
L_FP = PseudoDifferentialOperator(L_FP_symbol, [x], mode='symbol')
print("""
This operator has several important properties:
1. It generates a Markov semigroup: ρ(t) = exp(tL_FP)ρ(0)
2. The equilibrium distribution is the Gibbs state (Maxwell-Boltzmann)
3. It satisfies detailed balance (time-reversal symmetry)
4. The spectrum determines relaxation rates to equilibrium
""")
# Evolution towards equilibrium
t = symbols('t', real=True, positive=True)
print("\n" + "-"*70)
print("Time evolution operator exp(t L_FP)")
print("-"*70)
# For numerical values
gamma_val = 0.5
T_val = 1.0
L_FP_numeric = L_FP_symbol.subs([(gamma, gamma_val), (T, T_val)])
L_FP_num_op = PseudoDifferentialOperator(L_FP_numeric, [x], mode='symbol')
evolution_symbol = L_FP_num_op.exponential_symbol(t=t, order=4)
print(f"\nWith γ={gamma_val}, T={T_val}:")
print("exp(t L_FP) symbol (order 4):")
pprint(simplify(evolution_symbol))
# Relaxation time
print("\n" + "-"*70)
print("Relaxation to equilibrium")
print("-"*70)
print(f"""
The eigenvalues of L_FP determine relaxation timescales:
For this model:
- Largest eigenvalue: λ₀ = 0 (equilibrium state)
- Gap: λ₁ ≈ -γ (sets relaxation time τ ≈ 1/γ)
With γ = {gamma_val}:
- Relaxation time: τ ≈ {1/gamma_val:.2f}
""")
# Visualization of relaxation
from scipy.special import hermite as scipy_hermite
t_vals = np.linspace(0, 5/gamma_val, 100)
xi_vals = np.linspace(-4, 4, 200)
# Initial distribution: non-equilibrium packet
rho_0 = lambda xi: np.exp(-(xi - 2)**2)
# Equilibrium distribution: Maxwell-Boltzmann
rho_eq = lambda xi: np.exp(-xi**2/(2*T_val)) / np.sqrt(2*np.pi*T_val)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
# Time evolution
for i, t_val in enumerate([0, 1/gamma_val, 3/gamma_val, 5/gamma_val]):
    if i < 4:
        # Approximation: exponential decay towards equilibrium
        alpha = np.exp(-gamma_val * t_val)
        rho_t = lambda xi, a=alpha: a * rho_0(xi) + (1-a) * rho_eq(xi)

        ax_idx = (i//2, i%2)
        axes[ax_idx].plot(xi_vals, rho_0(xi_vals), 'b--', alpha=0.5, label='Initial')
        axes[ax_idx].plot(xi_vals, rho_t(xi_vals), 'r-', linewidth=2, label=f't={t_val:.2f}')
        axes[ax_idx].plot(xi_vals, rho_eq(xi_vals), 'g--', alpha=0.5, label='Equilibrium')
        axes[ax_idx].set_xlabel('ξ (momentum)')
        axes[ax_idx].set_ylabel('ρ(ξ, t)')
        axes[ax_idx].set_title(f'Distribution at t={t_val:.2f}')
        axes[ax_idx].legend()
        axes[ax_idx].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("\n✓ System relaxes to Maxwell-Boltzmann equilibrium")


## Example 10: Ergodic Theory - Transfer Operators

In [ ]:
x = symbols('x', real=True)
xi = symbols('xi', real=True)
print("""
CONTEXT: Ergodic Theory and Dynamical Systems
For a dynamical system with map T: M → M, the transfer operator
(Perron-Frobenius operator) acts on densities:
(L_T f)(x) = ∑_{T(y)=x} f(y)/|T'(y)|
For a smooth expanding map, L_T can be approximated by a
pseudo-differential operator.
Example: Baker's map (chaotic mixing)
T(x) = 2x mod 1
""")
# Transfer operator for the dilation T(x) = 2x
# L_T f(x) = (1/2)[f(x/2) + f((x+1)/2)]
# In terms of symbol, this corresponds to:
# Symbol approx: (1/2)[exp(iξ/2) + exp(iξ/2)exp(iπξ)]
transfer_symbol = (exp(I*xi/2) + exp(I*xi/2) * exp(I*np.pi*xi)) / 2
Transfer_op = PseudoDifferentialOperator(transfer_symbol, [x], mode='symbol')
print("\nTransfer operator symbol (Baker's map):")
pprint(simplify(Transfer_op.symbol))
# Powers of the transfer operator
print("\n" + "-"*70)
print("Powers of transfer operator: L_T^n")
print("-"*70)
print("""
L_T^n describes the evolution after n iterations of the map.
Key properties:
- L_T is a contraction in appropriate spaces
- Spectrum: discrete eigenvalues + continuous spectrum
- Leading eigenvalue λ₀ = 1 (invariant measure)
- Spectral gap: |λ₁| < 1 → mixing
""")
# Compute L_T^2 via composition
LT_squared_symbol = Transfer_op.compose_asymptotic(Transfer_op, order=3)
print("\nL_T² symbol (order 3):")
pprint(simplify(LT_squared_symbol))
# Visualization of mixing
print("\n" + "-"*70)
print("Visualization: Mixing dynamics")
print("-"*70)
def bakers_map(x):
    """Baker's map: T(x) = 2x mod 1"""
    return (2 * x) % 1
# Initial distribution
x_vals = np.linspace(0, 1, 1000, endpoint=False)
rho_initial = np.exp(-100*(x_vals - 0.3)**2)  # Localized distribution
rho_initial /= np.trapz(rho_initial, x_vals)   # Normalize
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
rho_current = rho_initial.copy()
for n, ax in enumerate(axes.flat):
    if n < 6:
        ax.plot(x_vals, rho_current, 'b-', linewidth=2)
        ax.axhline(1.0, color='r', linestyle='--', alpha=0.5, label='Uniform (equilibrium)')
        ax.set_xlabel('x')
        ax.set_ylabel('ρ(x)')
        ax.set_title(f'After {n} iterations')
        ax.set_ylim([0, 5])
        ax.grid(True, alpha=0.3)
        if n == 0:
            ax.legend()

        # Apply transfer operator (simplified: interpolation)
        if n < 5:
            x_preimage1 = x_vals / 2
            x_preimage2 = (x_vals + 1) / 2

            rho_new = 0.5 * (np.interp(x_preimage1, x_vals, rho_current, period=1) +
                            np.interp(x_preimage2, x_vals, rho_current, period=1))

            rho_current = rho_new
plt.tight_layout()
plt.show()
print("\n✓ Distribution converges to uniform (invariant measure)")
print("✓ This demonstrates mixing in chaotic systems")


## Example 10: Spectral Theory - Self-Adjoint Operators

In [ ]:
x = symbols('x', real=True)
xi = symbols('xi', real=True)
lam = symbols('lambda', real=True)
print("""
CONTEXT: Pure Mathematics - Spectral Theory
For a self-adjoint operator A, the resolvent is:
R_λ(A) = (A - λI)^{-1}
The resolvent exists for λ not in the spectrum σ(A).
Spectral properties:
- Poles of R_λ → eigenvalues
- Branch cuts → continuous spectrum
- Residues → eigenprojections
""")
# Schrödinger operator with harmonic potential
# H = -d²/dx² + x²  → symbol: ξ² + x²
H_symbol = xi**2 + x**2
H_op = PseudoDifferentialOperator(H_symbol, [x], mode='symbol')
print("\nHarmonic oscillator operator H:")
pprint(H_op.symbol)
# Resolvent (A - λI)^{-1}
print("\n" + "-"*70)
print("Resolvent R_λ(H) = (H - λ)^{-1}")
print("-"*70)
# Symbol: (ξ² + x² - λ)^{-1}
resolvent_symbol = 1 / (xi**2 + x**2 - lam)
print("\nResolvent symbol:")
pprint(resolvent_symbol)
print("""
Spectral information:
For the harmonic oscillator H = ξ² + x²:
- Spectrum: σ(H) = {2n + 1 : n ∈ ℕ} (discrete, unbounded)
- Eigenvalues: λₙ = 2n + 1, n = 0, 1, 2, ...
- No continuous spectrum
The resolvent has simple poles at λ = 2n + 1.
""")
# Visualization of the spectrum via the resolvent
print("\n" + "-"*70)
print("Visualization: Resolvent norm ‖R_λ(H)‖")
print("-"*70)
from sympy import lambdify
from scipy.integrate import trapezoid as scipy_trapezoid
x_val = 0
xi_test = np.linspace(-3, 3, 100)
lambda_vals = np.linspace(0, 10, 500)
resolvent_norm = []
for lam_val in lambda_vals:
    # Evaluate |R_λ(x, ξ)| for a typical ξ
    res_func = lambdify(xi, resolvent_symbol.subs([(x, x_val), (lam, lam_val)]), 'numpy')

    try:
        res_vals = res_func(xi_test)
        # Approximate L² norm
        norm = np.sqrt(scipy_trapezoid(np.abs(res_vals)**2, xi_test))
        resolvent_norm.append(norm)
    except:
        resolvent_norm.append(np.nan)
plt.figure(figsize=(12, 6))
plt.semilogy(lambda_vals, resolvent_norm, 'b-', linewidth=2)
# Mark theoretical eigenvalues
eigenvalues = [2*n + 1 for n in range(5)]
for ev in eigenvalues:
    plt.axvline(ev, color='r', linestyle='--', alpha=0.7, linewidth=1)
    plt.text(ev, plt.ylim()[1]*0.5, f'λ={ev}', rotation=90, va='bottom')
plt.xlabel('λ')
plt.ylabel('‖R_λ(H)‖ (log scale)')
plt.title('Resolvent Norm - Peaks at Eigenvalues')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("\n✓ Peaks in ‖R_λ‖ reveal the spectrum!")
print("✓ This is the foundation of spectral analysis")


## Example 11: Harmonic Analysis - Convolution Operators

In [ ]:
x = symbols('x', real=True)
xi = symbols('xi', real=True)
print("""
CONTEXT: Harmonic Analysis
A convolution operator with kernel k(x) acts as:
(K * f)(x) = ∫ k(x - y) f(y) dy
By Fourier theory, the symbol is simply:
p(ξ) = k̂(ξ) (Fourier transform of kernel)
This is independent of x (translation-invariant).
""")
# Example: Gaussian kernel (heat kernel at time t=1)
# k(x) = exp(-x²/2) / √(2π)
# k̂(ξ) = exp(-ξ²/2)
gaussian_kernel_symbol = exp(-xi**2 / 2)
Gauss_conv = PseudoDifferentialOperator(gaussian_kernel_symbol, [x], mode='symbol')
print("\nGaussian convolution operator symbol:")
pprint(Gauss_conv.symbol)
# Semigroup property
print("\n" + "-"*70)
print("Semigroup property: K_t * K_s = K_{t+s}")
print("-"*70)
# For the heat kernel: k_t(x) = exp(-x²/(4t)) / √(4πt)
# Symbol: exp(-tξ²)
t1, t2 = symbols('t1 t2', real=True, positive=True)
K_t1_symbol = exp(-t1 * xi**2)
K_t2_symbol = exp(-t2 * xi**2)
K_t1 = PseudoDifferentialOperator(K_t1_symbol, [x], mode='symbol')
K_t2 = PseudoDifferentialOperator(K_t2_symbol, [x], mode='symbol')
# Composition K_t1 ∘ K_t2
composition = K_t1.compose_asymptotic(K_t2, order=3)
print(f"\nK_t1 ∘ K_t2 =")
pprint(simplify(composition))
print(f"\nExpected: K_(t1+t2) = exp(-(t1+t2)ξ²)")
expected_composition = exp(-(t1 + t2) * xi**2)
pprint(expected_composition)
if simplify(composition - expected_composition) == 0:
    print("\n✓ Semigroup property verified!")
    print("✓ This is exact because convolution operators commute")
print("""
Applications in harmonic analysis:
1. Calderon-Zygmund theory (singular integrals)
2. Littlewood-Paley decomposition
3. Function spaces (Sobolev, Besov, Triebel-Lizorkin)
4. Multiplier theorems
5. Pseudodifferential calculus
""")
